# CRPSLoss & Probabilistic Forecasting

This notebook explains how to use `CRPSLoss` in BaseAttentive for
full-distribution probabilistic forecasting — going beyond point estimates
and simple quantile intervals.

## What is CRPS?

The **Continuous Ranked Probability Score (CRPS)** is a proper scoring
rule that measures how well a predicted probability distribution matches
the observed value.  Unlike:

- **MSE / MAE** — only score a single point prediction,
- **Pinball / quantile loss** — scores individual quantile levels separately,

CRPS rewards *calibrated* distributions that are both sharp (narrow) and
centred on the truth.  It collapses to MAE for a deterministic forecast.

$$
\text{CRPS}(F, y) = \int_{-\infty}^{\infty} \left(F(z) - \mathbf{1}[z \geq y]\right)^2 dz
$$

## Three Modes in BaseAttentive

| Mode | How it works | Best for |
|------|-------------|----------|
| `"quantile"` | Pinball loss averaged over user-specified quantile levels | Any output head; fast |
| `"gaussian"` | Closed-form CRPS for a single Gaussian predictive distribution | Unimodal, near-Gaussian targets |
| `"mixture"` | Monte-Carlo CRPS estimate from a Gaussian mixture | Multi-modal / heavy-tailed targets |

## Setup

In [ ]:
import numpy as np
import os

# Optional: pick a specific Keras backend before importing
# os.environ["KERAS_BACKEND"] = "torch"   # or "tensorflow" / "jax"

from base_attentive import BaseAttentive
from base_attentive.components.losses import CRPSLoss

print("CRPSLoss imported successfully")

## Synthetic Dataset

We create a small dataset with three distinct input streams (static,
dynamic, future) and a two-dimensional forecast target.

In [ ]:
BATCH      = 64
LOOKBACK   = 24   # historical time steps
HORIZON    = 12   # forecast horizon
STATIC_DIM = 4
DYN_DIM    = 8
FUT_DIM    = 4
OUTPUT_DIM = 2

rng = np.random.default_rng(42)

x_static  = rng.standard_normal((BATCH, STATIC_DIM)).astype("float32")
x_dynamic = rng.standard_normal((BATCH, LOOKBACK, DYN_DIM)).astype("float32")
x_future  = rng.standard_normal((BATCH, HORIZON, FUT_DIM)).astype("float32")

# Ground-truth targets: (batch, horizon, output_dim)
y_true = rng.standard_normal((BATCH, HORIZON, OUTPUT_DIM)).astype("float32")

print("x_static :", x_static.shape)
print("x_dynamic:", x_dynamic.shape)
print("x_future :", x_future.shape)
print("y_true   :", y_true.shape)

---

## Mode 1 — `"quantile"` (Pinball Loss)

The `quantile` mode computes the **pinball loss** (also called the
quantile loss or check function) for every quantile level you specify,
then averages across levels and all other dimensions.

### Build a quantile model

Pass `quantiles` to `BaseAttentive`; the output shape becomes
`(batch, horizon, n_quantiles, output_dim)`.

In [ ]:
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]

model_q = BaseAttentive(
    static_input_dim=STATIC_DIM,
    dynamic_input_dim=DYN_DIM,
    future_input_dim=FUT_DIM,
    output_dim=OUTPUT_DIM,
    forecast_horizon=HORIZON,
    quantiles=QUANTILES,
    embed_dim=32,
    num_heads=4,
    dropout_rate=0.1,
    name="QuantileModel",
)

# Forward pass to determine output shape
preds_q = model_q([x_static, x_dynamic, x_future])
print("Quantile output shape:", preds_q.shape)
# Expected: (64, 12, 5, 2) — batch, horizon, quantiles, output_dim

In [ ]:
# CRPSLoss in quantile mode
crps_q = CRPSLoss(mode="quantile", quantiles=QUANTILES)

# Compile and do a quick training step
model_q.compile(optimizer="adam", loss=crps_q)

history_q = model_q.fit(
    x=[x_static, x_dynamic, x_future],
    y=y_true,
    epochs=3,
    batch_size=16,
    verbose=1,
)
print("\nFinal training loss (quantile):", history_q.history["loss"][-1])

### Reading quantile predictions

The output layout is `(batch, horizon, n_quantiles, output_dim)`.  To
extract the median (index 2 for the 0.5 quantile above):

In [ ]:
import numpy as np

preds_np = np.array(preds_q)  # detach from any backend tensor

median_idx = QUANTILES.index(0.5)
median_pred = preds_np[:, :, median_idx, :]  # (64, 12, 2)

lower_idx = QUANTILES.index(0.1)
upper_idx = QUANTILES.index(0.9)
lower_bound = preds_np[:, :, lower_idx, :]   # (64, 12, 2)
upper_bound = preds_np[:, :, upper_idx, :]   # (64, 12, 2)

interval_width = upper_bound - lower_bound

print("Median prediction shape:", median_pred.shape)
print("80 %% PI width (mean over batch & horizon):",
      interval_width.mean().round(4))

---

## Mode 2 — `"gaussian"` (Closed-form)

In Gaussian mode the model predicts **mean** and **log-variance** for
each output.  `CRPSLoss` evaluates the exact closed-form CRPS:

$$
\text{CRPS}(\mathcal{N}(\mu, \sigma^2), y)
= \sigma \left[
    \frac{y - \mu}{\sigma} \left(2\Phi\!\left(\tfrac{y-\mu}{\sigma}\right) - 1\right)
    + 2\phi\!\left(\tfrac{y-\mu}{\sigma}\right)
    - \frac{1}{\sqrt{\pi}}
  \right]
$$

The model output shape is `(batch, horizon, 2 * output_dim)` where the
first `output_dim` channels are means and the last are log-variances.

In [ ]:
model_g = BaseAttentive(
    static_input_dim=STATIC_DIM,
    dynamic_input_dim=DYN_DIM,
    future_input_dim=FUT_DIM,
    output_dim=OUTPUT_DIM,
    forecast_horizon=HORIZON,
    output_mode="gaussian",   # ← enables mean+log-var head
    embed_dim=32,
    num_heads=4,
    dropout_rate=0.1,
    name="GaussianModel",
)

preds_g = model_g([x_static, x_dynamic, x_future])
print("Gaussian output shape:", preds_g.shape)
# Expected: (64, 12, 4) — batch, horizon, 2 * output_dim (mean + log-var)

In [ ]:
crps_g = CRPSLoss(mode="gaussian")

model_g.compile(optimizer="adam", loss=crps_g)
history_g = model_g.fit(
    x=[x_static, x_dynamic, x_future],
    y=y_true,
    epochs=3,
    batch_size=16,
    verbose=1,
)
print("\nFinal training loss (gaussian):", history_g.history["loss"][-1])

### Extracting mean and standard deviation

In [ ]:
import numpy as np

preds_g_np = np.array(preds_g)  # (64, 12, 4)

# Split along last axis: first half = mean, second half = log-var
mu_pred  = preds_g_np[..., :OUTPUT_DIM]       # (64, 12, 2)
logv_pred = preds_g_np[..., OUTPUT_DIM:]      # (64, 12, 2)
sigma_pred = np.exp(0.5 * logv_pred)          # standard deviation

print("Mean shape :", mu_pred.shape)
print("Sigma shape:", sigma_pred.shape)
print("Mean sigma (first batch element, first step):", sigma_pred[0, 0])

---

## Mode 3 — `"mixture"` (Gaussian Mixture MC)

The `mixture` mode fits a **Gaussian mixture model (GMM)** and estimates
CRPS via Monte Carlo sampling.  The model predicts, per output step:

- `K` component means — shape `(B, H, K, D)`
- `K` component log-variances — shape `(B, H, K, D)`  
- `K` mixing logits — shape `(B, H, K, 1)`

This handles **multi-modal** and **heavy-tailed** forecast distributions
(e.g., energy price spikes, flood events).

> **Apple Silicon note**: On MPS (M1/M2/M3) the mixture path uses a
> pure native-PyTorch implementation that keeps every tensor on CPU,
> bypassing any `keras.ops` call that could silently re-dispatch to MPS.
> This is the fix introduced in v2.0.1.

In [ ]:
N_COMPONENTS = 3   # number of Gaussian mixture components

model_m = BaseAttentive(
    static_input_dim=STATIC_DIM,
    dynamic_input_dim=DYN_DIM,
    future_input_dim=FUT_DIM,
    output_dim=OUTPUT_DIM,
    forecast_horizon=HORIZON,
    output_mode="mixture",
    n_mixture_components=N_COMPONENTS,
    embed_dim=32,
    num_heads=4,
    dropout_rate=0.1,
    name="MixtureModel",
)

preds_m = model_m([x_static, x_dynamic, x_future])
# preds_m is a tuple: (mu, log_var, logits)
if isinstance(preds_m, (list, tuple)):
    mu, log_var, logits = preds_m
    print("mu     shape:", mu.shape)       # (64, 12, 3, 2)
    print("log_var shape:", log_var.shape) # (64, 12, 3, 2)
    print("logits shape:", logits.shape)   # (64, 12, 3, 1)
else:
    print("Output shape:", preds_m.shape)

In [ ]:
crps_m = CRPSLoss(
    mode="mixture",
    n_mixture_components=N_COMPONENTS,
    mc_samples=128,      # number of Monte Carlo samples per forward pass
)

model_m.compile(optimizer="adam", loss=crps_m)
history_m = model_m.fit(
    x=[x_static, x_dynamic, x_future],
    y=y_true,
    epochs=3,
    batch_size=16,
    verbose=1,
)
print("\nFinal training loss (mixture):", history_m.history["loss"][-1])

---

## Comparing the Three Modes

| | `quantile` | `gaussian` | `mixture` |
|-|-----------|-----------|----------|
| **Distribution shape** | Any (implicit) | Unimodal | Multi-modal |
| **Computation** | Pinball sum | Closed-form | Monte Carlo |
| **Output extra dims** | `n_quantiles` | `× 2` (mean + log-var) | `K × (2 + 1/D)` |
| **Training speed** | Fast | Fast | Slower (MC samples) |
| **Calibration** | Quantile-level | Gaussian assumption | Flexible |
| **Apple Silicon (v2.0.1)** | ✅ | ✅ | ✅ (native PyTorch path) |

## Choosing `mc_samples`

Higher `mc_samples` gives a lower-variance CRPS estimate but increases
memory and compute.  Typical values:

| Situation | Recommended |
|-----------|-------------|
| Quick experiment | 32–64 |
| Production training | 128–256 |
| Final evaluation | 512–1024 |

In [ ]:
# Evaluate on hold-out set with higher mc_samples
crps_eval = CRPSLoss(
    mode="mixture",
    n_mixture_components=N_COMPONENTS,
    mc_samples=512,
)

model_m.compile(optimizer="adam", loss=crps_eval)
eval_loss = model_m.evaluate(
    x=[x_static, x_dynamic, x_future],
    y=y_true,
    verbose=0,
)
print(f"Evaluation CRPS (mixture, 512 samples): {eval_loss:.6f}")

## Custom Training Loop with CRPSLoss

For maximum control (e.g., gradient clipping, custom schedulers) you
can call `CRPSLoss` directly inside a `GradientTape`:

In [ ]:
# Example shown for the Torch backend — adapt for TF/JAX as needed.
# import torch
# optimizer = torch.optim.AdamW(model_q.parameters(), lr=1e-3)
#
# for epoch in range(5):
#     optimizer.zero_grad()
#     preds = model_q([x_static_t, x_dynamic_t, x_future_t])
#     loss  = crps_q(y_true_t, preds)
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(model_q.parameters(), 1.0)
#     optimizer.step()
#     print(f"Epoch {epoch+1}  loss={loss.item():.4f}")
print("See comment above for a Torch GradientTape-style training loop.")

---

## Next Steps

- [07_v2_spec_registry.ipynb](07_v2_spec_registry.ipynb) — declarative `BaseAttentiveSpec` and custom component registration
- [05_kernel_robust_networks.ipynb](05_kernel_robust_networks.ipynb) — kernel-robust training strategies
- [Full documentation](https://base-attentive.readthedocs.io/)
- [API reference: CRPSLoss](https://base-attentive.readthedocs.io/en/latest/api_reference.html)